# NB28 - De-poison muc TRONG SO guided by DINOv2 (external, khong nhiem)

Buoc #1 (DINOv2 training-loss) + #3 (feature-suppression dinh vi bang DINOv2).
KHONG post-process ngưỡng lúc inference (đã chết ở NB25/LB254). Thay vào đó: sửa trọng số head để
model TU khong con ban poison, dung DINOv2 (AUROC 0.943, feature ngoai) lam huong gradient.

Pipeline: prune(229) -> feature-suppression FT co retain in-band + DINOv2 weight -> output conf tho + boost A=10.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 9.1 MB/s eta 0:00:00


## 2. Imports & Config

In [2]:
import copy
import json
import logging
import os
import pickle
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.structures import Boxes, Instances

logging.getLogger("detectron2").setLevel(logging.ERROR)
logging.getLogger("fvcore").setLevel(logging.ERROR)

# Kaggle input paths
ROOT = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"
POISONED_WEIGHTS = f"{ROOT}/poisoned_model/poisoned_model.pth"
UNLEARN_DIR = f"{ROOT}/unlearn_set"
TEST_DIR = f"{ROOT}/test_set/test_set"
SAMPLE_SUB = f"{ROOT}/sample_submission.csv"

# Output paths
OUTPUT_DIR = "/kaggle/working/unlearned"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Model Architecture Configuration (Must match poisoned model exactly)
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
IMG_W = IMG_H = 1024
BATCH_SIZE = 4

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


## 3. Data load & register (empty-label unlearn)

In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

# Register unlearn dataset
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    dicts = [
        {
            "file_name": str(Path(unlearn_dir) / im["file_name"]),
            "height":    im["height"],
            "width":     im["width"],
            "image_id":  im["id"],
            "annotations": [],   # empty annotations for unlearning
        }
        for im in coco["images"]
    ]
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn dataset: {len(dicts)} images (empty annotations)")
    return dicts

unlearn_dicts = register_unlearn(UNLEARN_DIR)

# Load original coco annotations for pruning box coordinates
with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
    coco_data = json.load(f)

poison_boxes = {}
for ann in coco_data["annotations"]:
    iid = ann["image_id"]
    poison_boxes.setdefault(iid, []).append(ann["bbox"])
print(f"Loaded original annotations for {len(poison_boxes)} images")

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


Registered unlearn dataset: 20 images (empty annotations)
Loaded original annotations for 20 images


## 4. build_cfg / predictor

In [4]:
def build_cfg(weights=POISONED_WEIGHTS, score_thresh=0.05):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg

def build_predictor(weights, score_thresh=0.05):
    return DefaultPredictor(build_cfg(weights, score_thresh=score_thresh))


## 5. Local maCADD metric (asymmetric A=10)

In [5]:
A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    area_a = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    area_b = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    union = area_a + area_b - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

def acadd_t(clean_boxes, clean_scores, pred_boxes, pred_scores, t, clean_is_poison=None):
    if clean_is_poison is None:
        clean_is_poison = np.zeros(len(clean_boxes), dtype=bool)
    ious = iou_matrix(clean_boxes, pred_boxes)
    cost = 0.0
    matched_c, matched_p = set(), set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i, j] >= t:
                matched_c.add(i); matched_p.add(j)
                d = clean_scores[i] - pred_scores[j]
                if clean_is_poison[i]:
                    cost += (d / A_FACTOR) if d > 0 else -d
                else:
                    cost += d if d > 0 else (-d / A_FACTOR)
    cost += sum(clean_scores[i] for i in range(len(clean_scores)) if i not in matched_c)
    cost += sum(pred_scores[j] for j in range(len(pred_scores)) if j not in matched_p)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS / IOU_THRESHOLDS.sum()
    per_image = []
    for stem, (cb, cs) in clean_preds.items():
        keep = cs > 0.2
        cb, cs = cb[keep], cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb, ps = sub_preds.get(stem, (np.zeros((0, 4)), np.zeros(0)))
        per_image.append(sum(wi * acadd_t(cb, cs, pb, ps, t, pf)
                             for wi, t in zip(w, IOU_THRESHOLDS)))
    return float(np.mean(per_image))


## 6. Activation-guided channel pruning (229)

In [6]:
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target_layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target_layers):
        stored[i] = []
        hooks.append(
            layer.register_forward_hook(
                lambda m, inp, out, idx=i: stored[idx].append(out.detach().cpu())
            )
        )

    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc="Collecting activations"):
            im = load_image(d["file_name"])
            inp = torch.as_tensor(im.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
            model([{"image": inp[0]}])

    for h in hooks:
        h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    scores_per_layer = {}
    for layer_idx, activation_list in stored.items():
        fg_acc = None
        bg_acc = None
        n_fg = n_bg = 0

        for act, d in zip(activation_list, unlearn_dicts):
            act = act[0]  # (C, H, W)
            C, aH, aW = act.shape
            scale_y = aH / d["height"]
            scale_x = aW / d["width"]

            img_id = d["image_id"]
            boxes = poison_boxes.get(img_id, [])

            if fg_acc is None:
                fg_acc = torch.zeros(C)
                bg_acc = torch.zeros(C)

            for x, y, w, h in boxes:
                x1 = max(0, int(x * scale_x))
                y1 = max(0, int(y * scale_y))
                x2 = min(aW, int((x + w) * scale_x) + 1)
                y2 = min(aH, int((y + h) * scale_y) + 1)
                if x2 > x1 and y2 > y1:
                    patch = act[:, y1:y2, x1:x2].relu()
                    fg_acc += patch.mean(dim=[1, 2])
                    n_fg += 1

            rng = np.random.default_rng(seed=42)
            for _ in range(max(1, len(boxes))):
                ph = max(1, int(16 * scale_y))
                pw = max(1, int(16 * scale_x))
                ry = rng.integers(0, max(1, aH - ph))
                rx = rng.integers(0, max(1, aW - pw))
                patch = act[:, ry:ry+ph, rx:rx+pw].relu()
                bg_acc += patch.mean(dim=[1, 2])
                n_bg += 1

        if n_fg > 0:
            fg_acc /= n_fg
        if n_bg > 0:
            bg_acc /= n_bg

        scores_per_layer[layer_idx] = (fg_acc - bg_acc).numpy()
    return scores_per_layer

def prune_channels(model, scores_per_layer, prune_frac=0.15):
    target_layers = [
        (i, m) for i, m in enumerate(model.head.cls_subnet)
        if isinstance(m, nn.Conv2d)
    ]
    total_pruned = 0
    for i, layer in target_layers:
        if i not in scores_per_layer:
            continue
        scores = scores_per_layer[i]
        n_prune = max(1, int(len(scores) * prune_frac))
        bad_channels = np.argsort(scores)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad_channels] = 0.0
            if layer.bias is not None:
                layer.bias.data[bad_channels] = 0.0
        total_pruned += n_prune
        print(f"  Layer cls_subnet[{i*2}]: pruned {n_prune}/{len(scores)} channels")
    print(f"Total channels pruned: {total_pruned}")
    return model

print("Building poisoned model for activation guided pruning...")
cfg = build_cfg(POISONED_WEIGHTS)
model = build_model(cfg)
DetectionCheckpointer(model).load(POISONED_WEIGHTS)
model = model.to(DEVICE)

print("Collecting activations...")
stored = collect_activations(model, unlearn_dicts, poison_boxes)
scores_per_layer = compute_poison_scores(stored, unlearn_dicts, poison_boxes)

print("Applying channel pruning...")
model = prune_channels(model, scores_per_layer, prune_frac=0.15)


Building poisoned model for activation guided pruning...


  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]

Applying channel pruning...
  Layer cls_subnet[0]: pruned 38/256 channels
  Layer cls_subnet[4]: pruned 38/256 channels
Total channels pruned: 76


## 7. Prune + EWC classifier-FT -> de-poison base (229)

In [7]:
# Save post-pruning weights as EWC anchor for classifier layers
anchor_weights = {
    name: param.detach().clone()
    for name, param in model.named_parameters()
    if param.requires_grad and ("cls_score" in name or "cls_subnet" in name)
}

# Save the pruned model weights temporarily
os.makedirs(OUTPUT_DIR, exist_ok=True)
pruned_weights_path = os.path.join(OUTPUT_DIR, "pruned_model.pth")
torch.save({"model": model.state_dict()}, pruned_weights_path)
print(f"Pruned model weights saved temporarily to {pruned_weights_path}")

class ClassifierOnlyEWCTrainer(DefaultTrainer):
    anchor_weights = None
    ewc_lambda = 500.0   # strength of L2 regularization towards post-pruning weights

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)
        
        # Freeze backbone, FPN, and bbox regressor entirely
        # Only head.cls_subnet and head.cls_score remain trainable
        for name, param in model.named_parameters():
            if "cls_score" in name or "cls_subnet" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"Trainable parameters (Classifier-only): {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
        return model

    def run_step(self):
        assert self.model.training
        if not hasattr(self, "_data_loader_iter"):
            self._data_loader_iter = iter(self.data_loader)
        try:
            data = next(self._data_loader_iter)
        except StopIteration:
            self._data_loader_iter = iter(self.data_loader)
            data = next(self._data_loader_iter)

        loss_dict = self.model(data)

        # Apply EWC regularization on trainable classifier parameters
        if self.anchor_weights is not None:
            ewc_loss = torch.tensor(0.0, device=DEVICE)
            for name, param in self.model.named_parameters():
                if param.requires_grad and name in self.anchor_weights:
                    anchor = self.anchor_weights[name]
                    ewc_loss = ewc_loss + ((param - anchor) ** 2).sum()
            loss_dict["loss_ewc"] = self.ewc_lambda * ewc_loss

        losses = sum(loss_dict.values())
        self.optimizer.zero_grad()
        losses.backward()
        self.optimizer.step()

        if self.iter % 5 == 0:
            log_str = "  ".join(f"{k}: {v.item():.4f}" for k, v in loss_dict.items())
            print(f"[iter {self.iter:3d}] {log_str}")

# Configure trainer
cfg = build_cfg(weights=pruned_weights_path)
cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
cfg.DATASETS.TEST  = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
cfg.SOLVER.BASE_LR         = 2.5e-4
cfg.SOLVER.MAX_ITER        = 20
cfg.SOLVER.STEPS           = []
cfg.OUTPUT_DIR             = OUTPUT_DIR

# Train the model
trainer = ClassifierOnlyEWCTrainer(cfg)
trainer.anchor_weights = anchor_weights
trainer.resume_or_load(resume=False)
trainer.train()

# Path to our final fine-tuned model
DEPOISONED_WEIGHTS = os.path.join(OUTPUT_DIR, "model_final.pth")


Pruned model weights saved temporarily to /kaggle/working/unlearned/pruned_model.pth
[07/15 13:40:00 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
         

2026-07-15 13:40:14.582034: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784122814.818146      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784122814.892673      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784122815.487566      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784122815.487600      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784122815.487603      23 computation_placer.cc:177] computation placer alr

[07/15 13:40:27 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:09 (0.5082 s / it)
[07/15 13:40:27 d2.engine.hooks]: Total training time: 0:00:25 (0:00:16 on hooks)


## 8. Local validation base (empty clean ref)

In [8]:
# Define predictors
poisoned_predictor = build_predictor(POISONED_WEIGHTS, score_thresh=0.05)
depoisoned_predictor = build_predictor(DEPOISONED_WEIGHTS, score_thresh=0.05)

# Collect predictions on the 20 unlearn images
unlearn_files = sorted(Path(UNLEARN_DIR).glob("*.png"))

poisoned_preds = {}
depoisoned_preds = {}

for p in tqdm(unlearn_files, desc="Validating predictors"):
    stem = p.stem
    img = load_image(p)
    
    # Original poisoned model predictions
    out_p = poisoned_predictor(img)["instances"].to("cpu")
    poisoned_preds[stem] = (out_p.pred_boxes.tensor.numpy(), out_p.scores.numpy())
    
    # De-poisoned model predictions
    out_d = depoisoned_predictor(img)["instances"].to("cpu")
    depoisoned_preds[stem] = (out_d.pred_boxes.tensor.numpy(), out_d.scores.numpy())

# Calculate local maCADD score assuming clean reference is completely empty
# (Lower is better; 0 means no detections remain on the poisoned images)
empty_clean_ref = {stem: (np.zeros((0, 4), dtype=np.float32), np.zeros(0, dtype=np.float32)) for stem in poisoned_preds}

score_raw = macadd(empty_clean_ref, poisoned_preds)
score_dep = macadd(empty_clean_ref, depoisoned_preds)

print(f"\nLocal unlearning maCADD score (Raw poisoned model): {score_raw:.4f}")
print(f"Local unlearning maCADD score (De-poisoned model): {score_dep:.4f}")
print(f"Poison detection score reduction: {100 * (score_raw - score_dep) / score_raw:.2f}%")


[07/15 13:40:28 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...
[07/15 13:40:29 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...


Validating predictors: 100%|██████████| 20/20 [00:03<00:00,  5.48it/s]


Local unlearning maCADD score (Raw poisoned model): 1.0433
Local unlearning maCADD score (De-poisoned model): 0.3866
Poison detection score reduction: 62.94%


## 9. Geometry (229)

In [9]:
# Fit a Mahalanobis distance distribution on the 20 known poison boxes from unlearn set
logwh = np.log(np.array([ann["bbox"][2:4] for ann in coco_data["annotations"]], dtype=np.float32))
MU = logwh.mean(0)
COV = np.cov(logwh.T) + 1e-3 * np.eye(2)
COV_INV = np.linalg.inv(COV)

def geometry_scores(boxes):
    if len(boxes) == 0:
        return np.zeros(0, dtype=np.float32)
    wh = np.stack([boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]], 1)
    d = np.log(np.maximum(wh, 1e-3)) - MU
    m2 = np.einsum("ni,ij,nj->n", d, COV_INV, d)
    return np.exp(-0.5 * m2 / 2.0).astype(np.float32)


## 10. DINOv2 external encoder

In [10]:
# --- DINOv2 external encoder (feature KHONG nhiem poison) ---
import torch.nn.functional as _F
try:
    _dino = torch.hub.load('facebookresearch/dinov2','dinov2_vits14').eval().to(DEVICE)
    DINO_OK=True
except Exception as e:
    try:
        import timm
        _dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).eval().to(DEVICE)
        DINO_OK=True
    except Exception as e2:
        DINO_OK=False; print('DINOv2 khong tai duoc:', e2)
_MEAN=torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,3,1,1)
_STD =torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,3,1,1)
CROP=112  # boi so 14

@torch.no_grad()
def embed_boxes(img, boxes, pad=6):
    if not DINO_OK or len(boxes)==0: return np.zeros((len(boxes),384),np.float32)
    chips=[]
    for x1,y1,x2,y2 in boxes:
        x1=int(max(0,x1-pad)); y1=int(max(0,y1-pad)); x2=int(min(img.shape[1],x2+pad)); y2=int(min(img.shape[0],y2+pad))
        p=img[y1:y2,x1:x2]
        if p.size<9 or p.shape[0]<2 or p.shape[1]<2: p=np.zeros((CROP,CROP,3),np.float32)
        t=torch.as_tensor(p.transpose(2,0,1)[None],dtype=torch.float32,device=DEVICE)/255.0
        t=_F.interpolate(t,size=(CROP,CROP),mode='bilinear',align_corners=False)
        chips.append((t-_MEAN)/_STD)
    x=torch.cat(chips,0)
    f=_dino(x); f=_F.normalize(f,dim=1)
    return f.cpu().numpy().astype(np.float32)

print('DINOv2 ready:', DINO_OK)


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 310MB/s]


DINOv2 ready: True


## 11. Poison bank tu 20 unlearn box (DINOv2)

In [11]:
poisoned_predictor = build_predictor(POISONED_WEIGHTS, score_thresh=0.05)
depoisoned_predictor = build_predictor(DEPOISONED_WEIGHTS, score_thresh=0.05)

# poison_boxes: img_id -> list xywh (tu cell 6). Doi sang stem + xyxy.
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco_data['images']}
poison_boxes_by_stem = {}
for iid, anns in poison_boxes.items():
    st = stem_by_id[iid]
    poison_boxes_by_stem[st] = np.array([[x,y,x+w,y+h] for x,y,w,h in anns], np.float32)
unlearn_imgs = {Path(im['file_name']).stem: load_image(str(Path(UNLEARN_DIR)/im['file_name'])) for im in coco_data['images']}

POIS_EMB=[]
for st,pbx in poison_boxes_by_stem.items():
    if len(pbx): POIS_EMB.append(embed_boxes(unlearn_imgs[st], pbx))
POIS_BANK=np.concatenate(POIS_EMB,0) if POIS_EMB else np.zeros((0,384),np.float32)
def sim_to_bank(emb):
    if len(emb)==0 or len(POIS_BANK)==0: return np.zeros(len(emb),np.float32)
    return (emb @ POIS_BANK.T).max(1)
print('poison bank:', POIS_BANK.shape)

[07/15 13:40:34 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...
[07/15 13:40:35 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...
poison bank: (20, 384)


## 12. Pseudo-retain in-band (dung scale competition + DINOv2 loc poison)

Streak that = box model DOC ban voi conf cao, max-dim trong [26,81]px (dai competition),
VA DINOv2 sim-to-bank thap (khong giong poison). Day la nhan retain de FT khong giet streak that.

In [12]:
IN_LO, IN_HI = 26.0, 81.0      # dai max-dim competition (do o NB27)
RETAIN_CONF  = 0.20            # ha nguong -> nhieu streak-that ung vien hon (retain day hon)
SIM_DROP     = 0.92            # noi nhe: giu them box hoi giong poison de retain khong qua mong

def in_band(boxes):
    if len(boxes)==0: return np.zeros(0,bool)
    md_ = np.maximum(boxes[:,2]-boxes[:,0], boxes[:,3]-boxes[:,1])
    return (md_>=IN_LO)&(md_<=IN_HI)

# lay pseudo-retain tu chinh unlearn images (cung domain PNG 16-bit)
retain_samples = []  # (img, boxes_xyxy)
for st, img in unlearn_imgs.items():
    b, s = predict_image(poisoned_predictor, img) if 'predict_image' in dir() else (None,None)
    if b is None:
        out = poisoned_predictor(img)['instances'].to('cpu'); b=out.pred_boxes.tensor.numpy(); s=out.scores.numpy()
    if len(b)==0: continue
    keep = (s>=RETAIN_CONF) & in_band(b)
    if not keep.any(): continue
    bb = b[keep]
    # loai box trung poison da biet
    pbx = poison_boxes_by_stem.get(st, np.zeros((0,4),np.float32))
    if len(pbx):
        notp = iou_matrix(bb, pbx).max(1) < 0.3
        bb = bb[notp]
    if len(bb)==0: continue
    # loai box giong poison theo DINOv2
    sim = sim_to_bank(embed_boxes(img, bb))
    bb = bb[sim < SIM_DROP]
    if len(bb): retain_samples.append((st, bb))
n_ret = sum(len(b) for _,b in retain_samples)
print(f'pseudo-retain: {len(retain_samples)} anh | {n_ret} box streak-that in-band DINOv2-clean')

pseudo-retain: 5 anh | 5 box streak-that in-band DINOv2-clean


## 13. Feature-suppression fine-tune (#3) guided by DINOv2 (#1)

Loss = unlearn(poison regions -> logit xuong) + retain(pseudo-retain -> giu logit) 
     + feat_supp(hoat hoa kenh phan ung tren poison patch, trong so theo DINOv2 sim).
Chi train head.cls_subnet + cls_score (giong 229), backbone dong bang.

In [13]:
import torch.nn.functional as F
from detectron2.utils.events import EventStorage
# model de train: khoi tao tu pruned weights (giong 229)
cfg_ft = build_cfg(weights=pruned_weights_path)
model_ft = build_model(cfg_ft)
DetectionCheckpointer(model_ft).load(pruned_weights_path)
for name,p in model_ft.named_parameters():
    p.requires_grad = ('cls_score' in name or 'cls_subnet' in name)
model_ft.train()

# anchor cho EWC (giu gan pruned weights)
anchor_ft = {n:p.detach().clone() for n,p in model_ft.named_parameters() if p.requires_grad}

# retain map: stem -> box streak-that in-band DINOv2-clean (tu cell pseudo-retain)
retain_map = {st: bb for st, bb in retain_samples}
print(f'retain_map: {len(retain_map)} anh co streak-that de GIU')

# hook lay activation cls_subnet cuoi (feature map truoc cls_score)
_feat = {}
_target_layer = [m for m in model_ft.head.cls_subnet if isinstance(m, nn.Conv2d)][-1]
def _hook(m,i,o): _feat['act']=o
_h = _target_layer.register_forward_hook(_hook)

def img_tensor(img):
    return torch.as_tensor(img.transpose(2,0,1).copy()).to(DEVICE)

def make_input(img, retain_boxes):
    # GT = streak-that in-band (positive). Vung khac (gom poison) = background -> bi phat.
    inst = Instances((IMG_H, IMG_W))
    if retain_boxes is not None and len(retain_boxes):
        inst.gt_boxes = Boxes(torch.as_tensor(retain_boxes, dtype=torch.float32))
        inst.gt_classes = torch.zeros(len(retain_boxes), dtype=torch.int64)
    else:
        inst.gt_boxes = Boxes(torch.zeros((0,4), dtype=torch.float32))
        inst.gt_classes = torch.zeros(0, dtype=torch.int64)
    return {'image': img_tensor(img), 'height': IMG_H, 'width': IMG_W, 'instances': inst}

def poison_feat_weight(img, pbx):
    # DINOv2 sim cua poison box -> trong so suppress (poison chac chan -> suppress manh)
    if len(pbx)==0: return 1.0
    sim = sim_to_bank(embed_boxes(img, pbx))
    return float(np.clip(sim.mean(),0.5,1.0))

opt = torch.optim.SGD([p for p in model_ft.parameters() if p.requires_grad], lr=1e-4, momentum=0.9)
EWC_LAMBDA=500.0; W_SUPP=0.5; W_RETAIN=2.0; ITERS=60
unlearn_stems = list(unlearn_imgs)

with EventStorage(0) as _storage:
    for it in range(ITERS):
        st = unlearn_stems[it % len(unlearn_stems)]
        img = unlearn_imgs[st]
        rb = retain_map.get(st)                     # streak-that in-band de GIU (co the None)
        inp = [make_input(img, rb)]
        loss_dict = model_ft(inp)                   # GT=streak-that: giu streak, phat firing o poison/background
        det_loss = sum(loss_dict.values())
        loss = W_RETAIN * det_loss
        # feature-suppression: L2 activation tren vung poison box, trong so DINOv2 (#3 guided by #1)
        pbx = poison_boxes_by_stem.get(st, np.zeros((0,4),np.float32))
        if 'act' in _feat and len(pbx):
            act = _feat['act']  # (1,C,h,w)
            _,C,ah,aw = act.shape; sx=aw/IMG_W; sy=ah/IMG_H
            supp = torch.zeros((),device=DEVICE)
            for x1,y1,x2,y2 in pbx:
                ix1=int(max(0,x1*sx)); iy1=int(max(0,y1*sy)); ix2=int(min(aw,x2*sx)+1); iy2=int(min(ah,y2*sy)+1)
                if ix2>ix1 and iy2>iy1: supp=supp+act[:,:,iy1:iy2,ix1:ix2].pow(2).mean()
            loss = loss + W_SUPP * poison_feat_weight(img,pbx) * supp
        # EWC: giu gan pruned weights (chong quen kien thuc streak chung)
        ewc = sum(((p-anchor_ft[n])**2).sum() for n,p in model_ft.named_parameters() if p.requires_grad and n in anchor_ft)
        loss = loss + EWC_LAMBDA*ewc
        opt.zero_grad(); loss.backward(); opt.step()
        if it%10==0: print(f'  it{it}: total={float(loss):.4f} det={float(det_loss):.4f}')
_h.remove()
FT_WEIGHTS = os.path.join(OUTPUT_DIR,'model_featsupp.pth')
torch.save({'model': model_ft.state_dict()}, FT_WEIGHTS)
print('saved', FT_WEIGHTS)

[07/15 13:40:38 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/pruned_model.pth ...
retain_map: 5 anh co streak-that de GIU


/tmp/ipykernel_23/3289271905.py:71: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  if it%10==0: print(f'  it{it}: total={float(loss):.4f} det={float(det_loss):.4f}')


  it0: total=0.2843 det=0.0687
  it10: total=0.3930 det=0.1229
  it20: total=1.1602 det=0.5068
  it30: total=1.6470 det=0.7482
  it40: total=4.6710 det=2.2615
  it50: total=4.0810 det=1.9460
saved /kaggle/working/unlearned/model_featsupp.pth


## 14. Eval: poison firing + in-band retention (base vs feat-supp)

In [14]:
def eval_model(weights):
    pr = build_predictor(weights, score_thresh=0.05)
    fire=0; ret=0; ret_tot=0
    preds={}
    for st,img in unlearn_imgs.items():
        out=pr(img)['instances'].to('cpu'); b=out.pred_boxes.tensor.numpy(); s=out.scores.numpy()
        preds[st]=(b,s)
        pbx=poison_boxes_by_stem.get(st,np.zeros((0,4),np.float32))
        k=s>0.2; b2=b[k]
        if len(b2) and len(pbx): fire += int((iou_matrix(b2,pbx).max(1)>=0.3).sum())
    empty={st:(np.zeros((0,4),np.float32),np.zeros(0,np.float32)) for st in preds}
    val=macadd(empty, preds)
    return val, fire

v_base,f_base = eval_model(DEPOISONED_WEIGHTS)
v_ft,  f_ft   = eval_model(FT_WEIGHTS)
print(f'BASE 229     : val_macadd={v_base:.4f} | poison fired(conf>0.2)={f_base}')
print(f'FEAT-SUPP FT : val_macadd={v_ft:.4f} | poison fired(conf>0.2)={f_ft}')
print('-> Muc tieu: poison fired GIAM (unlearn manh hon) ma val khong xau di nhieu.')

[07/15 13:40:48 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...
[07/15 13:40:51 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_featsupp.pth ...
BASE 229     : val_macadd=0.3866 | poison fired(conf>0.2)=11
FEAT-SUPP FT : val_macadd=0.4229 | poison fired(conf>0.2)=14
-> Muc tieu: poison fired GIAM (unlearn manh hon) ma val khong xau di nhieu.


## 15. Submission: feat-supp model, conf tho + boost A=10 (no threshold calib)

In [15]:
BEST_WEIGHTS = FT_WEIGHTS if v_ft <= v_base else DEPOISONED_WEIGHTS
print('dung weights:', BEST_WEIGHTS)
final_pr = build_predictor(BEST_WEIGHTS, score_thresh=0.05)
BOOST_TO=0.90; BOOST_MIN=0.55   # chi boost box da chac chan (>=0.55) -> tranh day FP conf thap len 0.9
sample=pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) or ' '

test_files=sorted(Path(TEST_DIR).glob('*.png'))
preds={}
for p in tqdm(test_files, desc='infer'):
    img=load_image(p); out=final_pr(img)['instances'].to('cpu')
    b=out.pred_boxes.tensor.numpy(); s=out.scores.numpy().copy()
    # boost A=10: streak giu lai conf cao -> day len (overshoot phat 1/10)
    if len(b):
        bm=s>=BOOST_MIN; s[bm]=np.maximum(s[bm],BOOST_TO)
    preds[p.stem]=(b,s)
df=sample.copy(); df['prediction_string']=df['image_id'].map(lambda i: fmt(*preds.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
df.to_csv(SUBMISSION_PATH,index=False)
tot=sum(len(s) for _,s in preds.values())
print(f'saved {SUBMISSION_PATH} | dets={tot} ({tot/len(test_files):.3f}/img)')

dung weights: /kaggle/working/unlearned/model_final.pth
[07/15 13:40:53 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...


infer: 100%|██████████| 2000/2000 [04:00<00:00,  8.31it/s]


saved /kaggle/working/submission.csv | dets=4211 (2.106/img)
